# BiCyc Multi-Adapter — Huấn luyện trên Kaggle

Notebook này chạy pipeline **KeepLoRA + BiCyc + PFD routing** (Hướng 1) trên GPU Kaggle thay vì máy local:
- T4 16GB / P100 16GB (miễn phí).
- Bật **AMP fp16** + **TF32** để tăng tốc ~1.5–2x so với fp32 thuần.
- CIFAR-100 tự tải về `/kaggle/working/data/` lần chạy đầu.

## Chuẩn bị code (một trong hai cách)
**Cách A (khuyên dùng, không cần Internet):** zip toàn bộ repo rồi upload lên Kaggle dạng *Dataset* tên `bicyc-multiadapter` → mount tại `/kaggle/input/bicyc-multiadapter`.

**Cách B:** bật Internet (*Settings → Internet*) và điền `REPO_URL` ở cell dưới.

## Chọn GPU
*Settings → Accelerator → GPU T4 x2* hoặc *GPU P100*.

In [ ]:
# 1) Lấy code về /kaggle/working/repo
import shutil
from pathlib import Path

REPO_URL = ""  # Cách B: vd "https://github.com/<user>/BiCyc_MultiAdapter.git"

WORK_REPO = Path("/kaggle/working/repo")
candidates = [
    Path("/kaggle/input/bicyc-multiadapter"),
    Path("/kaggle/input/bicyc_multiadapter"),
    Path("/kaggle/input/BiCyc-MultiAdapter"),
    Path("/kaggle/input/BiCyc_MultiAdapter"),
]
# Tự động quét toàn bộ /kaggle/input tìm thư mục chứa pyproject.toml
if Path("/kaggle/input").exists():
    for p in Path("/kaggle/input").rglob("pyproject.toml"):
        candidates.append(p.parent)

repo = next((c for c in candidates if (c / "pyproject.toml").exists()), None)
if repo is None and REPO_URL:
    import subprocess
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORK_REPO)], check=True)
    repo = WORK_REPO
assert repo is not None, (
    "Khong tim thay repo. Upload repo len Kaggle Datasets (ten: bicyc-multiadapter)"
    " hoac dien REPO_URL o tren va bat Internet."
)
if repo.resolve() != WORK_REPO.resolve():
    if WORK_REPO.exists():
        shutil.rmtree(WORK_REPO)
    shutil.copytree(
        repo,
        WORK_REPO,
        ignore=shutil.ignore_patterns(".git", "__pycache__", ".venv", "data", "outputs", ".pytest_cache"),
    )
print("Repo san sang:", WORK_REPO)


In [ ]:
# 2) Cài dependencies và cho repo vào PYTHONPATH mà KHÔNG dùng editable install
import os
import sys

repo_src = str(WORK_REPO / "src")
if repo_src not in sys.path:
    sys.path.insert(0, repo_src)
existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = repo_src + (os.pathsep + existing_pythonpath if existing_pythonpath else "")
%pip install -q -r {WORK_REPO}/requirements/base.txt
import bicyc_multiadapter
print("bicyc_multiadapter", bicyc_multiadapter.__version__)

In [ ]:
# 3) Kiểm tra GPU
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))
else:
    print("CANH BAO: chua bat GPU! Settings -> Accelerator -> GPU T4/P100.")

In [ ]:
# 3b) Chuẩn bị dữ liệu CIFAR-100 trên Kaggle
import os
import shutil
import subprocess
import sys
import tarfile
from pathlib import Path

repo_src = str(WORK_REPO / "src")
existing_pythonpath = os.environ.get("PYTHONPATH", "")
os.environ["PYTHONPATH"] = (
    repo_src + (os.pathsep + existing_pythonpath if existing_pythonpath else "")
)

# 1. Kiểm tra xem người dùng có attach sẵn Dataset CIFAR-100 trong /kaggle/input không
kaggle_input = Path("/kaggle/input")
candidates = [
    kaggle_input / "cifar-100-python",
    kaggle_input / "cifar100" / "cifar-100-python",
    kaggle_input / "cifar-100" / "cifar-100-python",
]
if kaggle_input.exists():
    for p in kaggle_input.rglob("cifar-100-python"):
        if p.is_dir() and any(p.iterdir()):
            candidates.append(p)

found_input_data = next((c for c in candidates if c.is_dir() and any(c.iterdir())), None)

DATA_ROOT = Path("/kaggle/working/data/cifar100")
DATA_ROOT.mkdir(parents=True, exist_ok=True)
extracted_dir = DATA_ROOT / "cifar-100-python"

if found_input_data is not None:
    if not extracted_dir.exists():
        try:
            os.symlink(found_input_data, extracted_dir)
            print(f"✅ Đã tạo symlink từ Kaggle Input: {found_input_data} -> {extracted_dir}")
        except Exception:
            shutil.copytree(found_input_data, extracted_dir)
            print(f"✅ Đã copy từ Kaggle Input: {found_input_data} -> {extracted_dir}")
    else:
        print(f"✅ Dữ liệu đã sẵn sàng tại: {extracted_dir}")
elif extracted_dir.is_dir() and any(extracted_dir.iterdir()):
    print(f"✅ Đã có sẵn dữ liệu đã giải nén tại: {extracted_dir}")
else:
    # Tìm kiếm file nén trong /kaggle/input hoặc trong /kaggle/working/data
    tar_candidates = (
        list(kaggle_input.rglob("cifar-100-python.tar*"))
        + list(kaggle_input.rglob("*.tar*"))
        + list(DATA_ROOT.glob("*.tar*"))
        + list(DATA_ROOT.glob("*.tgz"))
    )
    if tar_candidates:
        tar_path = tar_candidates[0]
        print(f"📦 Tìm thấy file nén: {tar_path}. Đang giải nén vào {DATA_ROOT}...")
        with tarfile.open(tar_path, "r:*") as tar:
            tar.extractall(path=DATA_ROOT)
        print(f"✅ Giải nén thành công vào: {DATA_ROOT}")
    else:
        print("⬇️ Tải CIFAR-100 lần đầu (~170MB)...")
        env = dict(os.environ, PYTHONPATH=os.environ["PYTHONPATH"])
        subprocess.run(
            [sys.executable, "-m", "bicyc_multiadapter.data.prepare", "--root", str(DATA_ROOT)],
            cwd=str(WORK_REPO),
            env=env,
            check=True,
        )
        if not extracted_dir.is_dir():
            for tar_path in DATA_ROOT.glob("*.tar*"):
                print(f"📦 Đang giải nén {tar_path.name}...")
                with tarfile.open(tar_path, "r:*") as tar:
                    tar.extractall(path=DATA_ROOT)

print("DATA_ROOT =", str(DATA_ROOT))


## Cấu hình thí nghiệm

| Preset | Khi nào dùng |
| --- | --- |
| `keeplora_bicyc` | Đề xuất đầy đủ (routed multi-adapter + BiCyc + adaptive gate), batch 128 |
| `keeplora_bicyc_8gb` | Như trên nhưng batch 32 + AMP bật sẵn (tối ưu nhất cho T4/P100 16GB) |
| `keeplora_original` | Baseline KeepLoRA nguyên gốc (merge sau task, không distillation) |

Với T4/P100 16 GB: khuyến nghị dùng `keeplora_bicyc_8gb` (batch 32) hoặc `keeplora_bicyc` (batch 128).

> ⏱️ **Thời gian ước tính**: ViT-B/16 @224 × 20 epochs × 10 tasks mất khoảng 2.5 - 4 giờ trên GPU T4 (đã bật AMP fp16).
> Bạn có thể đặt `EPOCHS_PER_TASK = 1` để chạy thử toàn bộ 10 tasks (smoke test mất ~5-10 phút) trước khi chạy thật.
>
> 💡 **Hỗ trợ Tự động Resume**: Nếu phiên bị ngắt kết nối hoặc hết hạn thời gian Kaggle (9-12h), pipeline tự động lưu snapshot `checkpoint_live.pt` (mỗi epoch) và `checkpoint_boundary.pt` (mỗi task). Bạn chỉ cần chạy lại notebook với cùng `OUT_DIR` để tiếp tục từ đúng epoch/task đang dở mà không bị mất tiến trình.


In [ ]:
# 4) Tham số run — sửa ở đây thay vì sửa yaml
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # giảm phân mảnh VRAM

EXPERIMENT = "keeplora_bicyc_8gb"  # T4/P100 16GB: dung keeplora_bicyc_8gb (batch 32)
SEED = 2024
EPOCHS_PER_TASK = 20               # đặt 1 để smoke test nhanh toàn bộ pipeline
BATCH_SIZE = 32                    # T4/P100 16GB: 32 (an toan khong OOM)
USE_AMP = True                     # fp16 mixed precision (hiệu quả nhất trên T4/RTX)
NUM_WORKERS = 2                    # Kaggle chỉ có 2-4 vCPU
ACTIVATION_CACHE_ROWS = 4096       # giảm xuống 1536 nếu hết RAM CPU
CHECKPOINT_EVERY_EPOCHS = 1        # lưu snapshot mỗi N epoch để resume khi hết thời gian (ghi đè, ~350MB)

OUT_DIR = f"/kaggle/working/outputs/{EXPERIMENT}/seed_{SEED}"

overrides = [
    f"experiment={EXPERIMENT}",
    f"experiment.seed={SEED}",
    f"data.root={DATA_ROOT}",
    f"data.num_workers={NUM_WORKERS}",
    f"experiment.data.root={DATA_ROOT}",
    f"experiment.data.num_workers={NUM_WORKERS}",
    f"output_dir={OUT_DIR}",
    f"experiment.activation_cache_rows={ACTIVATION_CACHE_ROWS}",
    f"experiment.train.amp={str(USE_AMP).lower()}",
    f"experiment.checkpoint_every_epochs={CHECKPOINT_EVERY_EPOCHS}",
]
if EPOCHS_PER_TASK is not None:
    overrides.append(f"experiment.train.epochs_per_task={EPOCHS_PER_TASK}")
if BATCH_SIZE is not None:
    overrides.append(f"experiment.train.batch_size={BATCH_SIZE}")
print("Hydra overrides:\n  " + "\n  ".join(overrides))


In [ ]:
# 4b) (Tuỳ chọn) Khôi phục Checkpoint từ file ZIP / Dataset đã upload để Resume
# Dùng khi bạn mở phiên Kaggle mới và muốn train tiếp task đang dở từ phiên trước.
import os
import shutil
from pathlib import Path

out_path = Path(OUT_DIR)
out_path.mkdir(parents=True, exist_ok=True)

# Cách A: Nếu bạn upload checkpoint dạng Kaggle Dataset (vào /kaggle/input)
found_checkpoints = (
    list(Path("/kaggle/input").rglob("checkpoint_*.pt"))
    + list(Path("/kaggle/input").rglob("task_*.pt"))
)
found_logs = (
    list(Path("/kaggle/input").rglob("history.jsonl"))
    + list(Path("/kaggle/input").rglob("train_log.csv"))
    + list(Path("/kaggle/input").rglob("metrics.json"))
)

# Cách B: Nếu có file zip trong /kaggle/input hoặc /kaggle/working
zip_candidates = list(Path("/kaggle/input").rglob("*.zip")) + list(Path("/kaggle/working").glob("*.zip"))

if any(out_path.glob("*.pt")):
    print("Đã có sẵn checkpoint trong thư mục output:", [p.name for p in out_path.glob('*.pt')])
elif found_checkpoints:
    print(f"Tìm thấy {len(found_checkpoints)} checkpoint từ /kaggle/input, đang copy sang {out_path}...")
    for cp in found_checkpoints:
        shutil.copy2(cp, out_path / cp.name)
    for log_f in found_logs:
        if not (out_path / log_f.name).exists():
            shutil.copy2(log_f, out_path / log_f.name)
    print("Khôi phục thành công:", [p.name for p in out_path.glob('*.pt')])
elif zip_candidates:
    target_zip = zip_candidates[0]
    print(f"Đang giải nén checkpoint từ {target_zip} vào {out_path}...")
    shutil.unpack_archive(str(target_zip), out_path)
    print("Giải nén thành công:", [p.name for p in out_path.glob('*.pt')])
else:
    print("Chưa có checkpoint cũ nào. Mô hình sẽ huấn luyện mới từ Task 0.")


In [ ]:
# 5) HUẤN LUYỆN — Gọn gàng, chỉ báo cáo khi hoàn thành từng Epoch / Task kèm ETA
import os
import subprocess
import sys

repo_src = str(WORK_REPO / "src")
env = dict(
    os.environ,
    PYTHONPATH=repo_src
    + (
        os.pathsep + os.environ.get("PYTHONPATH", "")
        if os.environ.get("PYTHONPATH")
        else ""
    ),
    PYTHONUNBUFFERED="1",
    TQDM_DISABLE="1",  # Tắt tiến trình từng batch để tránh tràn log Kaggle
    TF_CPP_MIN_LOG_LEVEL="3",
)

cmd = [sys.executable, "-u", "-m", "bicyc_multiadapter.train", *overrides]
print("$", " ".join(cmd), "\n" + "=" * 80)

process = subprocess.Popen(
    cmd,
    cwd=str(WORK_REPO),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

try:
    for line in iter(process.stdout.readline, ""):
        if any(
            x in line
            for x in [
                "Unable to register cu",
                "external/local_xla",
                "cpu_feature_guard",
            ]
        ):
            continue
        print(line, end="", flush=True)

    process.stdout.close()
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(f"Training thất bại với mã lỗi {return_code}")
    print("\n" + "=" * 80)
    print("🎉 HUẤN LUYỆN TOÀN BỘ 10 TASKS THÀNH CÔNG!")
except KeyboardInterrupt:
    process.terminate()
    print("\n⚠️ Đã tạm dừng huấn luyện (Checkpoint đã được tự động lưu lại).")

## Đánh giá & trực quan kết quả

In [ ]:
# 6) Đọc metrics, vẽ accuracy matrix và xuất báo cáo tổng kết dạng text (summary_report.txt)
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

out_dir = Path(OUT_DIR)
metrics_file = out_dir / "metrics.json"
history_file = out_dir / "history.jsonl"

if metrics_file.exists():
    metrics = json.loads(metrics_file.read_text(encoding="utf-8"))
    summary = metrics.get("summary", {})
    raw_matrix = metrics.get("accuracy_matrix", [])
    num_tasks = len(raw_matrix)
    matrix = np.full((num_tasks, num_tasks), np.nan)
    for i, row in enumerate(raw_matrix):
        for j, val in enumerate(row):
            if j < num_tasks:
                matrix[i, j] = val
    
    # --- 1. Tạo file báo cáo tổng kết dạng text dễ đọc (summary_report.txt) ---
    report_lines = [
        "=" * 80,
        "                BICYC MULTI-ADAPTER EXPERIMENT REPORT (CIL 10 TASKS)",
        "=" * 80,
        f"Experiment : {EXPERIMENT}",
        f"Seed       : {SEED}",
        f"Output Dir : {OUT_DIR}",
        "-" * 80,
        "CIL CORE METRICS SUMMARY:",
        f"  * Last Average Accuracy (A_B)        : {summary.get('last_average', 0.0) * 100:.2f}% ({summary.get('last_average', 0.0):.4f})",
        f"  * Incremental Average Accuracy (A_bar): {summary.get('incremental_average', 0.0) * 100:.2f}% ({summary.get('incremental_average', 0.0):.4f})",
        f"  * Catastrophic Forgetting (F)         : {summary.get('forgetting', 0.0):.4f}",
        "-" * 80,
        "ACCURACY MATRIX (%):",
        "(Hang = sau khi hoc task i, Cot = do chinh xac tren task j)",
        "-" * 80,
    ]
    
    num_tasks = matrix.shape[0]
    header = "Task | " + " ".join(f"T{j:<5}" for j in range(num_tasks)) + " | Last_Avg"
    report_lines.append(header)
    report_lines.append("-" * len(header))
    
    for i in range(num_tasks):
        row_vals = []
        for j in range(num_tasks):
            if j <= i and not np.isnan(matrix[i, j]):
                row_vals.append(f"{matrix[i, j]*100:5.2f}%")
            else:
                row_vals.append("   -  ")
        valid_accs = [matrix[i, j] for j in range(i + 1) if not np.isnan(matrix[i, j])]
        row_avg = np.mean(valid_accs) * 100 if valid_accs else 0.0
        report_lines.append(f"T{i:02d}  | " + " ".join(row_vals) + f" | {row_avg:5.2f}%")
    
    report_lines.append("=" * 80)
    report_text = "\n".join(report_lines)
    
    report_path = out_dir / "summary_report.txt"
    report_path.write_text(report_text, encoding="utf-8")
    print(report_text)
    print(f"\nDa luu bao cao tong ket tai: {report_path}")
    
    # --- 2. Vẽ và lưu ảnh ma trận Accuracy ---
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(matrix, cmap="viridis", vmin=0, vmax=1)
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            if not np.isnan(matrix[i, j]):
                ax.text(j, i, f"{matrix[i, j]*100:.1f}%", ha="center", va="center", color="w", fontsize=8)
    ax.set_xlabel("Task da hoc (Seen Task)")
    ax.set_ylabel("Sau khi hoc Task (Evaluation Stage)")
    ax.set_title(f"{EXPERIMENT} (Seed {SEED})\nLast Avg: {summary.get('last_average', 0)*100:.2f}% | Forget: {summary.get('forgetting', 0):.4f}")
    fig.colorbar(im, ax=ax, label="Accuracy")
    plt.tight_layout()
    plt.savefig(out_dir / "accuracy_matrix.png", dpi=200)
    plt.show()
else:
    print(f"Chưa tìm thấy {metrics_file}. Hãy chắc chắn huấn luyện đã hoàn thành ít nhất Task 0.")


In [ ]:
# 7) Dọn dẹp checkpoint phụ và đóng gói kết quả nghiên cứu
import os
import shutil
from pathlib import Path

out_dir = Path(OUT_DIR)

# Nếu đã có các file task_XX.pt hoàn chỉnh, xóa bớt snapshot tạm để tiết kiệm dung lượng
if any(out_dir.glob("task_*.pt")):
    for rf in ["checkpoint_boundary.pt", "checkpoint_live.pt"]:
        f_path = out_dir / rf
        if f_path.exists():
            f_path.unlink()

archive = shutil.make_archive("/kaggle/working/results", "zip", out_dir)
print("=== DANH SÁCH CÁC FILE ĐƯỢC LƯU TRONG RESULTS.ZIP ===")
for path in sorted(out_dir.iterdir()):
    if path.is_file():
        print(f"  [File] {path.name:30s} ({path.stat().st_size / 1024:,.1f} KB)")
    elif path.is_dir():
        print(f"  [Dir ] {path.name:30s}")
print("\nĐã tạo thành công file tải về:", archive)


## Mẹo vận hành trên Kaggle

- **T4** hỗ trợ fp16 TensorCore → `USE_AMP = True` lợi ích lớn nhất; **P100** không có TF32 nhưng AMP vẫn giảm băng thông nhớ.
- Ablation nhanh bằng override trong `overrides`, ví dụ:
  - Tắt adaptive gate (λ_t ≡ 1): `experiment.alignment.adaptive_gate=false`
  - Baseline nguyên gốc: `experiment=keeplora_original`
  - Merge thay vì routed: `experiment.keeplora.merge_after_task=true`
  - Đổi seed: `experiment.seed=0` (chạy ≥ 3 seeds mỗi cấu hình khi báo cáo).
- Xem loss/metric chi tiết sau khi tải `results.zip`: `tensorboard --logdir outputs`.
- **Tự động resume**: pipeline lưu `checkpoint_boundary.pt` (sau mỗi task) và
  `checkpoint_live.pt` (mỗi `CHECKPOINT_EVERY_EPOCHS` epoch hoặc khi bấm interrupt). Nếu phiên bị
  ngắt, chỉ cần **chạy lại cell huấn luyện với cùng `OUT_DIR`** — training tiếp tục từ đúng epoch
  giữa task (kèm optimizer + RNG state), không mất công đã train. File cũ luôn bị ghi đè nên chỉ
  tốn dung lượng của 1 checkpoint.
- So sánh catastrophic forgetting: xem `history.jsonl` (mỗi task một dòng gồm per-task accuracy +
  last/incremental/forgetting tích lũy) và TensorBoard scalar `eval/running_forgetting`;
  loss từng epoch nằm trong `train_log.csv`.
- **Tái sử dụng dataset**: cell chuẩn bị dữ liệu tự dò CIFAR-100 trong `/kaggle/input` (nếu bạn
  attach sẵn một dataset `cifar-100-python`) hoặc tải về `/kaggle/working/data`. Bấm **Save Version**
  sau phiên đầu để thư mục data được giữ lại; các phiên sau không phải tải lại (~170MB).